In [3]:
# Libraries
import numpy as np 
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform, transform_bounds # Reprojection
from rasterio.transform import Affine
np.set_printoptions(suppress = True) # Turn off scientific notation
from tqdm import tqdm  

In [2]:
# Only run initially!!

#from zipfile import ZipFile

#land_cover_zip = r'./data/land_cover/Annual_NLCD_LndCov_2024_CU_C1V1.zip'
#land_cover_out = r'./data/land_cover'

#with ZipFile(land_cover_zip, 'r') as zObject:
    # Extract downloaded land cover data and store in data > land_cover folder
#    zObject.extractall(path = land_cover_out)

In [6]:
# View raw land cover data - check crs, profile, nodata, unique values (DONE - 5 min)
lc = r'./data/land_cover/Annual_NLCD_LndCov_2024_CU_C1V1.tif'

unique_values = set() # Create a set to store unique values

with rasterio.open(lc, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
    
    # Check unique values
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window)
        
        # Updates unique values to the set
        unique_values.update(np.unique(data)) 

# Print out unique values
print(f'Unique values: {sorted(unique_values)}')

Profile: {'driver': 'GTiff', 'dtype': 'uint8', 'nodata': 250.0, 'width': 160000, 'height': 105000, 'count': 1, 'crs': CRS.from_wkt('PROJCS["AEA        WGS84",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]'), 'transform': Affine(30.0, 0.0, -2415585.0,
       0.0, -30.0, 3314805.0), 'blockxsize': 512, 'blockysize': 512, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: 250.0
CRS: PROJCS["AEA        WGS84",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,

In [7]:
# View raw land cover data - check %nodata (DONE - 2 min)

lc = r'./data/land_cover/Annual_NLCD_LndCov_2024_CU_C1V1.tif'

with rasterio.open(lc, mode = 'r') as src:

    # Set total_nodata
    total_nodata = 0
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)

        # Count nodata cells
        total_nodata += np.sum(data.mask)
    
    # % nodata 
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

% nodata cells: 0.46558087315476193


In [8]:
# This is why the % nodata is so high - including the entire bounding box not just CONUS land!!!
with rasterio.open(lc) as src:
    print('Bounds:\n', src.bounds)
    print('Width:', src.width, 'Height:', src.height)

Bounds:
 BoundingBox(left=-2415585.0, bottom=164805.0, right=2384415.0, top=3314805.0)
Width: 160000 Height: 105000


In [15]:
# Update profile (DONE - 7 min)
    # dtype = int8 (int16 used as intermediary)
    # nodata = -10 (& replace nodata cells with new nodata value)

lc = './data/land_cover/Annual_NLCD_LndCov_2024_CU_C1V1.tif'
lc_profile_update = './data/land_cover/land_cover_profile_update.tif'

with rasterio.open(lc) as src:
    profile = src.profile.copy()
    profile.update(dtype = rasterio.int8,
                   nodata = -10)

    with rasterio.open(lc_profile_update, 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window).astype(rasterio.int16) # int16 as intermediary to accomodate 250 in the raw land cover
            
            # Replace 250 cells (previous nodata value) to -10 (new nodata value)
            data[data == 250] = -10
            
            # Write out new raster
            dst.write(data.astype(rasterio.int8), 1, window = window)

In [16]:
# Verify updated profile - check unique values (DONE - 5 min)

lc_profile_update = './data/land_cover/land_cover_profile_update.tif'

unique_values = set() # Create a set to store unique values

with rasterio.open(lc_profile_update, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
    
    # Check unique values
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window)
        
        # Updates unique values to the set
        unique_values.update(np.unique(data)) 

# Print out unique values
print(f'Unique values: {sorted(unique_values)}')

Profile: {'driver': 'GTiff', 'dtype': 'int8', 'nodata': -10.0, 'width': 160000, 'height': 105000, 'count': 1, 'crs': CRS.from_wkt('PROJCS["AEA        WGS84",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]'), 'transform': Affine(30.0, 0.0, -2415585.0,
       0.0, -30.0, 3314805.0), 'blockxsize': 512, 'blockysize': 512, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CRS: PROJCS["AEA        WGS84",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,2

In [2]:
# Reproject to epsg:5070; 30m cells 

lc_profile_update = './data/land_cover/land_cover_profile_update.tif'
lc_reprojected = './data/land_cover/land_cover_reprojected.tif' 

dst_crs = 'epsg:5070' # Target crs
res = 30 # Res in m

with rasterio.open(lc_profile_update) as src:
    # Calculate transform matrix for output
    dst_transform, dst_width, dst_height = calculate_default_transform(
        src.crs, 
        dst_crs, 
        src.width, 
        src.height, 
        *src.bounds, # Unpacks bounds (left, bottom, right, top)
        resolution = res
    )

    # Set output properties
    dst_profile = src.profile.copy()
    dst_profile.update(
        crs = dst_crs,
        transform = dst_transform,
        width = dst_width,
        height = dst_height,
        nodata = -10,
        tiled = True,
        blockxsize = 128,
        blockysize = 128
    )

    # Reproject each band
    with rasterio.open(lc_reprojected, 'w', **dst_profile) as dst:
        for i in range(1, src.count + 1):
            reproject(
                source = rasterio.band(src, i),
                destination = rasterio.band(dst, i),
                src_transform = src.transform,
                src_crs = src.crs,
                dst_transform = dst_transform,
                dst_crs = dst_crs,
                resampling = Resampling.nearest # Nearest resampling for categorical data
            )

In [3]:
# Verify reprojection and unique values

lc_reprojected = './data/land_cover/land_cover_reprojected.tif'

unique_values = set() # Create a set to store unique values

with rasterio.open(lc_reprojected, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
    
    # Check unique values
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window)
        
        # Updates unique values to the set
        unique_values.update(np.unique(data)) 

# Print out unique values
print(f'Unique values: {sorted(unique_values)}')

Profile: {'driver': 'GTiff', 'dtype': 'int8', 'nodata': -10.0, 'width': 160000, 'height': 105000, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2415585.0000220123,
       0.0, -30.0, 3314804.999958794), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'deflate', 'interleave': 'band'}
Nodata: -10.0
CR

In [4]:
# Verify % nodata after reprojection and profile update

lc_reprojected = './data/land_cover/land_cover_reprojected.tif'

with rasterio.open(lc_reprojected, mode = 'r') as src:

    # Set total_nodata
    total_nodata = 0
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)

        # Count nodata cells
        total_nodata += np.sum(data.mask)
    
    # % nodata 
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

% nodata cells: 0.46558087315476193


In [8]:
# Co-register land cover with dist to gwt raster (inches convert)

# Define co-register function - NEAREST RESAMPLING
def coregister_rasters(infile, match, outfile):
    """Reproject a file to match the shape and projection of existing raster. 
    
    Parameters
    ----------
    infile : (string) path to input file to reproject
    match : (string) path to raster with desired shape and projection 
    outfile : (string) path to output file tif
    """
    # Open input
    with rasterio.open(infile) as src:
        src_transform = src.transform
        
        # Open input to match
        with rasterio.open(match) as match:
            dst_crs = match.crs
            dst_transform = match.transform # Ensures resolutions of outfile and match will be exactly the same
            dst_width = match.width
            dst_height = match.height

        # Set properties for output
        dst_kwargs = src.meta.copy()
        dst_kwargs.update({'crs': dst_crs,
                           'transform': dst_transform,
                           'width': dst_width,
                           'height': dst_height,
                           'nodata': -10})
        print('Coregistered to shape:', dst_height, dst_width,'\n Affine', dst_transform)
        
        # Open output
        with rasterio.open(outfile, "w", **dst_kwargs) as dst:
            # Iterate through bands and write using reproject function
            for i in range(1, src.count + 1):
                reproject(
                    source = rasterio.band(src, i),
                    destination = rasterio.band(dst, i),
                    src_transform = src.transform,
                    src_crs = src.crs,
                    dst_transform = dst_transform,
                    dst_crs = dst_crs,
                    resampling = Resampling.nearest)
                

# Apply coregister_rasters
lc_reprojected = './data/land_cover/land_cover_reprojected.tif' # Input
ref_raster = './data/SSURGO_raw/dist_GWT/gwt_inches.tif' # Match
lc_coregisterd = './data/land_cover/land_cover_coregistered.tif' # Output

coregister_rasters(
    infile = lc_reprojected,
    match = ref_raster,
    outfile = lc_coregisterd
)

Coregistered to shape: 96751 153996 
 Affine | 30.00, 0.00,-2356125.00|
| 0.00,-30.00, 3172575.00|
| 0.00, 0.00, 1.00|


In [9]:
# Verify co-registered raster 

lc_coregisterd = './data/land_cover/land_cover_coregistered.tif' # Output

unique_values = set() # Create a set to store unique values

with rasterio.open(lc_coregisterd, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
    
    # Check unique values
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window)
        
        # Updates unique values to the set
        unique_values.update(np.unique(data)) 

# Print out unique values
print(f'Unique values: {sorted(unique_values)}')

Profile: {'driver': 'GTiff', 'dtype': 'int8', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 153996, 'blockysize': 1, 'tiled': False, 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolution (30.0, 30.0)
Un

In [10]:
# Verify % nodata after co-register 

lc_coregisterd = './data/land_cover/land_cover_coregistered.tif' 

with rasterio.open(lc_coregisterd, mode = 'r') as src:

    # Set total_nodata
    total_nodata = 0
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)

        # Count nodata cells
        total_nodata += np.sum(data.mask)
    
    # % nodata 
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

% nodata cells: 0.39744363730039706


In [3]:
# Match raster with final HSG raster
hsg_final_composite = './data/SSURGO_raw/hsg/hsg_FINAL_composite.tif' 
lc_coregisterd = './data/land_cover/land_cover_coregistered.tif' 
lc_MATCH = './data/land_cover/land_cover_MATCH.tif' 

# Open hsg composite raster - want conus cells to match THIS raster
with rasterio.open(hsg_final_composite) as conus:

    # Nodata value for the conus raster
    conus_nodata = conus.nodata 
    # Profile conus raster
    profile = conus.profile.copy()
    
    # Open raster - want to convert any cells containing data where conus contains NODATA to nodata
    with rasterio.open(lc_coregisterd) as src:
        
        src_nodata = src.nodata # Nodata value 
        
        # Open output raster
        with rasterio.open(lc_MATCH, 'w', **profile) as dst:
        
            for ji, window in conus.block_windows(1):
            
                # hsg composite raster data
                conus_data = conus.read(1, window = window)
            
                # Land cover raster data
                src_data = src.read(1, window = window)
            
                # Identify cells where conus_data == nodata value
                remove_mask = (conus_data == conus_nodata)
            
                # Convert cells in src where conus is nodata to the nodata value
                src_data[remove_mask] = src_nodata
            
                # Write out
                dst.write(src_data, 1, window = window)

In [4]:
# Verify matched raster 

lc_MATCH = './data/land_cover/land_cover_MATCH.tif' 

unique_values = set() # Create a set to store unique values

with rasterio.open(lc_MATCH, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
    
    # Check unique values
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window)
        
        # Updates unique values to the set
        unique_values.update(np.unique(data)) 

# Print out unique values
print(f'Unique values: {sorted(unique_values)}')

Profile: {'driver': 'GTiff', 'dtype': 'int8', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolutio

In [5]:
# Verify % nodata after match

lc_MATCH = './data/land_cover/land_cover_MATCH.tif' 

with rasterio.open(lc_MATCH, mode = 'r') as src:

    # Set total_nodata
    total_nodata = 0
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)

        # Count nodata cells
        total_nodata += np.sum(data.mask)
    
    # % nodata 
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

% nodata cells: 0.4246642052725585


In [2]:
# Reclassify land cover classes to standard suitability scores:
    # 21, 22, 23, 24 - standard suitability score: 10
    # 81, 82 - standard suitability score: 4
    # 11, 12, 31, 41, 42, 43, 52, 71, 90, 95 - standard suitability score: 0

lc_MATCH = './data/land_cover/land_cover_MATCH.tif' 
lc_standardized = './data/land_cover/land_cover_standardized.tif' 
    
with rasterio.open(lc_MATCH, mode = 'r') as src:
    profile = src.profile.copy()
    profile.update(dtype = rasterio.float32)
    
    with rasterio.open(lc_standardized, 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window).astype(rasterio.float32)
            
            # Convert 21, 22, 23, 24 to 10
            standard_10 = np.isin(data, [21, 22, 23, 24])
            
            # Convert 81, 82 to 4
            standard_4 = np.isin(data, [81, 82])
            
            # Convert 11, 12, 31, 41, 42, 43, 52, 71, 90, 95 to 0
            standard_0 = np.isin(data, [11, 12, 31, 41, 42, 43, 52, 71, 90, 95])
            
            # Apply masks
            data[standard_10] = 10
            data[standard_4] = 4
            data[standard_0] = 0
            
            # Write out new raster
            dst.write(data, 1, window = window)

In [3]:
# Verify standardized values

lc_standardized = './data/land_cover/land_cover_standardized.tif' 

with rasterio.open(lc_standardized, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [4]:
# Verify standardized unique values 

lc_standardized = './data/land_cover/land_cover_standardized.tif' 

unique_values = set() # Create a set to store unique values

with rasterio.open(lc_standardized, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
    
    # Check unique values
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window)
        
        # Updates unique values to the set
        unique_values.update(np.unique(data)) 

# Print out unique values
print(f'Unique values: {sorted(unique_values)}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [2]:
# Reclassify land cover classes to urban and non-urban classes:
    # 21, 22, 23, 24 >> 1
    # 81, 82, 11, 12, 31, 41, 42, 43, 52, 71, 90, 95 >> 0

lc_MATCH = './data/land_cover/land_cover_MATCH.tif' 
lc_urban_nonurban = './data/land_cover/land_cover_urban_nonurban.tif' 
    
with rasterio.open(lc_MATCH, mode = 'r') as src:
    profile = src.profile.copy()
    profile.update(dtype = rasterio.float32)
    
    with rasterio.open(lc_urban_nonurban, 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window).astype(rasterio.float32)
            
            # Convert 21, 22, 23, 24 to 1 (urban land covers)
            urban = np.isin(data, [21, 22, 23, 24])
            
            # Convert 81, 82, 11, 12, 31, 41, 42, 43, 52, 71, 90, 95 to 0 (non urban land covers)
            nonurban = np.isin(data, [81, 82, 11, 12, 31, 41, 42, 43, 52, 71, 90, 95])
            
            # Apply masks
            data[urban] = 1
            data[nonurban] = 0
            
            # Write out new raster
            dst.write(data, 1, window = window)

In [5]:
# Calculate % urban and non-urban

lc_urban_nonurban = './data/land_cover/land_cover_urban_nonurban.tif' 

total_1 = 0
total_0 = 0
total_nodata = 0

with rasterio.open(lc_urban_nonurban) as src:

    # Total number of blocks
    total_blocks = sum(1 for _ in src.block_windows(1))
    print(f'Total blocks: {total_blocks}')

    for ji, window in tqdm(src.block_windows(1), total = total_blocks, desc = 'Block window processing'):
        data = src.read(1, window = window, masked = True)

        total_1 += np.count_nonzero(data == 1) # Total cells = 1
        total_0 += np.count_nonzero(data == 0) # Total cells = 0

        total_nodata += np.sum(data.mask) # Calculates the total number of nodata cells

    all_cells = src.width * src.height

    all_NON_nodata_cells = all_cells - total_nodata

    percent_1 = total_1 / all_NON_nodata_cells
    print(f'% percent urban: {percent_1}')
    
    percent_0 = total_0 / all_NON_nodata_cells
    print(f'% percent non-urban: {percent_0}')


    print(f'%s add to : {percent_1 + percent_0}')

Total blocks: 910224


Block window processing: 100%|██████████| 910224/910224 [05:21<00:00, 2830.20it/s]


% percent urban: 0.07053847757705753
% percent non-urban: 0.9294615224229424
%s add to : 1.0
